# Colab Setup
Use the next setup cells in order in Colab instead of the local-only cells below.

Setup order:
1. Bootstrap Miniconda and create the `openvla-oft` environment.
2. Install system packages and pinned PyTorch wheels.
3. Clone the repo and install it in editable mode.
4. Install the pinned FlashAttention wheel.
5. Verify the final software stack and GPU.

Key constraints:
- Do not use `pip install python=3.10`; Colab cannot change the interpreter that way.
- This repo's RLDS pipeline imports TensorFlow, so the install needs a real Python 3.10 environment for `tensorflow==2.15.0`.
- `torch==2.2.0` does not support Blackwell GPUs. If Colab assigns a Blackwell GPU, switch to an A100, L4, T4, or another supported runtime before training.

In [2]:
# check GPU and CUDA setup:

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

True
NVIDIA A100-SXM4-80GB
79.250732421875


In [ ]:
%%bash
set -euo pipefail

unset PYTHONPATH
cd /content

if [ ! -d /content/miniconda3 ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.1-0-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes --set changeps1 no
conda create -n openvla-oft python=3.10 || true
conda activate openvla-oft

which python
python --version
echo "$CONDA_DEFAULT_ENV"

In [ ]:
%%bash
set -euo pipefail

export DEBIAN_FRONTEND=noninteractive
unset PYTHONPATH
source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content

stdbuf -oL -eL apt-get update
stdbuf -oL -eL apt-get install -y rsync

stdbuf -oL -eL python -m pip install --upgrade pip setuptools wheel packaging ninja
stdbuf -oL -eL python -m pip install --progress-bar on \
  torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
  --index-url https://download.pytorch.org/whl/cu121

python -c "import torch; print(torch.__version__); print(torch.version.cuda)"

In [ ]:
%%bash
set -euo pipefail

unset PYTHONPATH
source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content

if [ ! -d /content/openvla-oft_human ]; then
  stdbuf -oL -eL git clone https://github.com/alexlee511/openvla-oft_human.git /content/openvla-oft_human
fi

cd /content/openvla-oft_human
stdbuf -oL -eL python -m pip install --progress-bar on -e .

In [ ]:
%%bash
set -euo pipefail

unset PYTHONPATH
source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content/openvla-oft_human

FLASH_ATTN_WHL=flash_attn-2.5.5+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
FLASH_ATTN_URL="https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.5/${FLASH_ATTN_WHL}"

stdbuf -oL -eL wget "$FLASH_ATTN_URL"
ls -lh "$FLASH_ATTN_WHL"
stdbuf -oL -eL python -m pip install --progress-bar on "./$FLASH_ATTN_WHL"

In [ ]:
%%bash
set -euo pipefail

unset PYTHONPATH
source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content/openvla-oft_human

python - <<'PY'
import importlib.metadata
import torch

print(f"Torch       : {torch.__version__} (CUDA {torch.version.cuda})")
print(f"TensorFlow  : {importlib.metadata.version('tensorflow')}")
print(f"flash-attn  : {importlib.metadata.version('flash_attn')}")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
print(f"GPU         : {gpu}")
if 'Blackwell' in gpu:
    raise SystemExit(
        'Torch 2.2.0 does not support Blackwell GPUs in this repo setup. '
        'Use a non-Blackwell Colab GPU or update the full training stack to a newer PyTorch/FlashAttention combination.'
    )
PY

## Colab Dataset Check And Sync
Run the next cell to inspect what is already present under `/content/openvla-oft_human`.
If `modified_libero_rlds/pure_ik` is missing, choose one of these options before training:
- Use the SSH/rsync cell and replace the source host, user, port, and remote path with the machine that actually stores your dataset.
- If your dataset is already in Google Drive, use the Drive copy cell instead.

In [ ]:
%%bash
set -euo pipefail

echo "[repo root]"
ls -la /content/openvla-oft_human || true

echo
echo "[modified_libero_rlds]"
ls -la /content/openvla-oft_human/modified_libero_rlds || true

# echo
# echo "[pure_ik]"
# ls -la /content/openvla-oft_human/modified_libero_rlds/pure_ik || true

# echo
# echo "[pure_ik datasets]"
# find /content/openvla-oft_human/modified_libero_rlds/pure_ik -maxdepth 1 -mindepth 1 -type d | sort || true

In [ ]:
# %%bash
# set -euo pipefail

# DEST_ROOT=/content/openvla-oft_human/modified_libero_rlds
# SOURCE_SUBDIR=pure_ik

# # Replace these with the SSH machine that actually stores your dataset.
# SOURCE_USER=your_user
# SOURCE_HOST=your.host.example.com
# SOURCE_PORT=22
# REMOTE_ROOT=/path/to/openvla-oft_human/modified_libero_rlds

# mkdir -p "$DEST_ROOT/$SOURCE_SUBDIR"

# echo "Syncing ${SOURCE_USER}@${SOURCE_HOST}:${REMOTE_ROOT}/${SOURCE_SUBDIR}/ -> ${DEST_ROOT}/${SOURCE_SUBDIR}/"
# echo "Edit SOURCE_USER, SOURCE_HOST, SOURCE_PORT, and REMOTE_ROOT before running."

# rsync -rlptDvz --progress --no-g \
#   -e "ssh -p ${SOURCE_PORT} -o StrictHostKeyChecking=accept-new" \
#   "${SOURCE_USER}@${SOURCE_HOST}:${REMOTE_ROOT}/${SOURCE_SUBDIR}/" \
#   "${DEST_ROOT}/${SOURCE_SUBDIR}/"

# echo
# echo "Sync complete. Current datasets under ${DEST_ROOT}/${SOURCE_SUBDIR}:"
# find "${DEST_ROOT}/${SOURCE_SUBDIR}" -maxdepth 1 -mindepth 1 -type d | sort || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -euo pipefail

# Example: copy dataset from Google Drive into the Colab workspace.
DRIVE_DATA_ROOT="/content/drive/MyDrive/openvla-oft_human/modified_libero_rlds"
DEST_ROOT="/content/openvla-oft_human/modified_libero_rlds"
SOURCE_SUBDIR="pure_ik"

mkdir -p "$DEST_ROOT"
rsync -rlptDvz --progress --no-g \
  "$DRIVE_DATA_ROOT/$SOURCE_SUBDIR/" \
  "$DEST_ROOT/$SOURCE_SUBDIR/"

echo
echo "Copy complete. Current datasets under ${DEST_ROOT}/${SOURCE_SUBDIR}:"
find "$DEST_ROOT/$SOURCE_SUBDIR" -maxdepth 1 -mindepth 1 -type d | sort || true

In [ ]:
%%bash
set -euo pipefail

unset PYTHONPATH
source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content/openvla-oft_human

DATA_ROOT=/content/openvla-oft_human/modified_libero_rlds/pure_ik
DATASET_NAME=libero_goal_humanized_no_noops

if [ -n "${WANDB_API_KEY:-}" ]; then
  wandb login --relogin "$WANDB_API_KEY"
else
  export WANDB_MODE=offline
  echo "WANDB_API_KEY is not set; running Weights & Biases in offline mode."
fi

if [ ! -d "$DATA_ROOT" ]; then
  echo "Missing data root: $DATA_ROOT"
  echo "This Colab clone does not include your RLDS dataset. Sync or mount the dataset first."
  exit 1
fi

if [ ! -d "$DATA_ROOT/$DATASET_NAME" ]; then
  echo "Missing dataset directory: $DATA_ROOT/$DATASET_NAME"
  echo "Available datasets under $DATA_ROOT:"
  ls -1 "$DATA_ROOT" || true
  exit 1
fi

echo "Using dataset directory: $DATA_ROOT/$DATASET_NAME"

torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py \
  --vla_path openvla/openvla-7b \
  --data_root_dir "$DATA_ROOT" \
  --dataset_name "$DATASET_NAME" \
  --run_root_dir runs/test/pure_ik/openvla-oft_humanized_libero_goal \
  --use_l1_regression True \
  --use_diffusion False \
  --use_film False \
  --num_images_in_input 2 \
  --use_proprio True \
  --batch_size 8 \
  --grad_accumulation_steps 8 \
  --learning_rate 5e-4 \
  --num_steps_before_decay 15000 \
  --max_steps 20000 \
  --save_freq 5000 \
  --save_latest_checkpoint_only False \
  --image_aug True \
  --lora_rank 32 \
  --wandb_entity alexlee511-national-taipei-university-of-technology \
  --wandb_project openvla-oft_human \
  --run_id_note pure_ik_parallel_dec--8_acts_chunk--continuous_acts--L1_regression--3rd_person_img--wrist_img--proprio_state

In [ ]:
# 如果你想盡量看到「還沒跑完時的即時輸出」，可以試這幾種方式。

# 第一，對 Python 程式加 unbuffered：
# PYTHONUNBUFFERED=1 torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py ...

# 或直接：
# python -u your_script.py

# 第二，用 stdbuf 減少緩衝：
# stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py ...

In [ ]:

%%bash
pip install python=3.10
sudo apt update
sudo apt install -y rsync
python -m pip install --upgrade pip
python -m pip install \
  torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
  --index-url https://download.pytorch.org/whl/cu121
git clone https://github.com/alexlee511/openvla-oft_human.git

cd openvla-oft_human/
pip install -e .

pip install -U pip setuptools wheel packaging ninja
ninja --version

wget "https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.5/flash_attn-2.5.5+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"


ls -lh flash_attn-2.5.5+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

pip install ./flash_attn-2.5.5+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

python - <<'PY'
import torch, importlib.metadata
print(f"Torch  : {torch.__version__}  (CUDA {torch.version.cuda})")
print("flash-attn:", importlib.metadata.version("flash_attn"))
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
PY


In [ ]:
%%bash
source ~/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /home/vsp1323/Humanized-VLA/openvla-oft_human

torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py \
  --vla_path openvla/openvla-7b \
  --data_root_dir modified_libero_rlds/pure_ik \
  --dataset_name libero_spatial_humanized_no_noops \
  --run_root_dir runs/pure_ik/openvla-oft_humanized_libero_spatial \
  --use_l1_regression True \
  --use_diffusion False \
  --use_film False \
  --num_images_in_input 2 \
  --use_proprio True \
  --batch_size 8 \
  --grad_accumulation_steps 8 \
  --learning_rate 5e-4 \
  --num_steps_before_decay 15000 \
  --max_steps 20000 \
  --save_freq 5000 \
  --save_latest_checkpoint_only False \
  --image_aug True \
  --lora_rank 32 \
  --wandb_entity alexlee511-national-taipei-university-of-technology \
  --wandb_project openvla-oft_human \
  --run_id_note pure_ik_parallel_dec--8_acts_chunk--continuous_acts--L1_regression--3rd_person_img--wrist_img--proprio_state

In [ ]:
# V100 Humanized:
%%bash
set -euo pipefail

source /content/miniconda3/etc/profile.d/conda.sh
conda activate openvla-oft
cd /content/openvla-oft_human
torchrun --standalone --nnodes 1 --nproc-per-node 1 vla-scripts/finetune.py \
  --vla_path openvla/openvla-7b \
  --data_root_dir /nfs/Workspace/Alex/openvla-oft_human/modified_libero_rlds \
  --dataset_name libero_10_joint_no_noops \
  --run_root_dir /nfs/Workspace/Alex/openvla-oft_human/runs/openvla-oft_joint_libero_10 \
  --use_l1_regression True \
  --use_diffusion False \
  --use_film False \
  --num_images_in_input 2 \
  --use_proprio True \
  --batch_size 2 \
  --grad_accumulation_steps 32 \
  --learning_rate 5e-4 \
  --num_steps_before_decay 15000 \
  --max_steps 20000 \
  --save_freq 10000 \
  --save_latest_checkpoint_only False \
  --merge_lora_during_training False \
  --image_aug True \
  --lora_rank 32 \
  --wandb_entity alexlee511-national-taipei-university-of-technology \
  --wandb_project openvla-oft \
  --run_id_note parallel_dec--8_acts_chunk--continuous_acts--L1_regression--3rd_person_img--wrist_img--proprio_state

(dataset)  
--data_root_dir /modified_libero_rlds/liu_ik

ALL:
--dataset_name libero_4_task_suites_humanized_no_noops

--dataset_name libero_10_humanized_no_noops  
--dataset_name libero_spatial_humanized_no_noops  
--dataset_name libero_goal_humanized_no_noops  
--dataset_name libero_object_humanized_no_noops  

(local)  
--run_root_dir /runs/liu_ik/openvla-oft_humanized_libero_4_tasks

--run_root_dir /runs/liu_ik/openvla-oft_humanized_libero_10  
--run_root_dir /runs/liu_ik/openvla-oft_humanized_libero_spatial  
--run_root_dir /runs/liu_ik/openvla-oft_humanized_libero_goal  
--run_root_dir /runs/liu_ik/openvla-oft_humanized_libero_object  

--run_id_note pure_ik_

(vcp)  
--run_root_dir /home/vsp1323/Humanized-VLA/openvla-oft_human/runs/liu_ik/openvla-oft_humanized_libero_4_tasks  

--run_root_dir /home/vsp1323/Humanized-VLA/openvla-oft_human/runs/liu_ik/openvla-oft_humanized_libero_10  
--run_root_dir /home/vsp1323/Humanized-VLA/openvla-oft_human/runs/liu_ik/openvla-oft_humanized_libero_spatial  
--run_root_dir /home/vsp1323/Humanized-VLA/openvla-oft_human/runs/liu_ik/openvla-oft_humanized_libero_goal  
--run_root_dir /home/vsp1323/Humanized-VLA/openvla-oft_human/runs/liu_ik/openvla-oft_humanized_libero_object  
